# Industrial Graded RAG implementation experiment

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

# Setup Confituration

In [44]:
class Config:
    # setup mistral configuration
    mistral_api_key = os.getenv("MISTRAL_API_KEY")
    mistral_chat_model = os.getenv("MISTRAL_CHAT_MODEL")
    mistral_embed_model = os.getenv("MISTRAL_EMBED_MODEL")
    mistral_embed_dimension = os.getenv("MISTRAL_EMBED_DIMENSION")

    # pinecone configuration
    pinecone_api_key = os.getenv("PINECONE_API_KEY")
    pinecone_index = os.getenv("PINECONE_INDEX")
    pinecone_namespace = os.getenv("PINECONE_NAMESPACE")
    pinecone_region = os.getenv("PINECONE_REGION")

# LLM Service setup

In [20]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

class MistralService:
    chatModel : ChatMistralAI
    embeddingModel : MistralAIEmbeddings

    def __init__(self):
        self.chatModel = self.connectMistralChatModel()
        self.embeddingModel = self.connectMistralEmbedModel()

    def connectMistralChatModel(self, model_name : str = Config.mistral_chat_model) -> ChatMistralAI :
        try:
            return ChatMistralAI(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
        
    def connectMistralEmbedModel(self, model_name: str = Config.mistral_embed_model) -> MistralAIEmbeddings :
        try:
            return MistralAIEmbeddings(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
    
    def getChatModel(self) -> ChatMistralAI:
        return self.chatModel
    
    def getEmbedModel(self) -> MistralAIEmbeddings:
        return self.embeddingModel

# Pinecone Setup

In [46]:
from pinecone import Pinecone, ServerlessSpec

class PineconeService:
    pc: Pinecone

    def __init__(self):
        self.pc = self.connectPinecone()

    def connectPinecone(self) -> Pinecone:
        pc = Pinecone()
        index_list_obj = pc.list_indexes()
        index_list = [index.name for index in index_list_obj]

        spec = ServerlessSpec(
            cloud = "aws",
            region =  Config.pinecone_region
        )

        for Config.pinecone_index in index_list_obj:
            try:
                pc.create_index(
                    name = Config.pinecone_namespace,
                    dimension = Config.mistral_embed_dimension,
                    metric = "cosine",
                    spec = spec
                )
            except Exception as e:
                print(f"Something went wrong in pinecone index creation...")
                raise
        
        return pc

    def getPinecone(self) -> Pinecone:
        return self.pc

In [28]:
pc = Pinecone()
print(pc.list_indexes())

[{
    "name": "industrial-rag-name-space",
    "metric": "cosine",
    "host": "industrial-rag-name-space-uicas0q.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null
}]
